# Madrid por distritos — análisis socioeconómico completo (INE 2023)

Recorre **todas** las variables como casos de estudio: cada bloque tiene su **visual** (gráfico o mapa) y su **conclusión**. Datos reales del INE (Atlas de Renta 2023) + geometría del Ayto. de Madrid.

**Requisito:** `python src/extract.py` (descarga los 4 CSV + el zip de distritos a `data/raw/`).

Ver el plan: `docs/PLAN_ANALISIS_DISTRITOS.md`.


## 0. Setup, carga y variables derivadas


In [ ]:
import sys, os, importlib
SRC = os.path.abspath('../src')
if SRC not in sys.path: sys.path.insert(0, SRC)
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config, load_ine, maps
importlib.reload(load_ine); importlib.reload(maps)

ZIP = str(config.DATA_RAW / 'distritos_madrid.zip')

# nombres reales de las columnas (del INE)
RENTA_NETA_HOGAR = 'Renta neta media por hogar'
RENTA_BRUTA_HOGAR = 'Renta bruta media por hogar'
MEDIANA = 'Mediana de la renta por unidad de consumo'
GINI = 'Índice de Gini'
P80P20 = 'Distribución de la renta P80/P20'
SALARIO = 'ingresos_Fuente de ingreso: salario'
PENSIONES = 'ingresos_Fuente de ingreso: pensiones'
PARO = 'ingresos_Fuente de ingreso: prestaciones por desempleo'
OTRAS_PREST = 'ingresos_Fuente de ingreso: otras prestaciones'
OTROS_ING = 'ingresos_Fuente de ingreso: otros ingresos'
EDAD = 'demo_Edad media de la población'
POB = 'demo_Población'
UNIPERSONAL = 'demo_Porcentaje de hogares unipersonales'
MAYORES65 = 'demo_Porcentaje de población de 65 y más años'
ESPANOLA = 'demo_Porcentaje de población española'
MENORES18 = 'demo_Porcentaje de población menor de 18 años'

# cargar datos reales
mad = load_ine.construir_madrid()

# --- variables DERIVADAS ---
def norm(s): return (s - s.min()) / (s.max() - s.min())
mad['peso_impuestos_pct'] = (mad[RENTA_BRUTA_HOGAR] - mad[RENTA_NETA_HOGAR]) / mad[RENTA_BRUTA_HOGAR] * 100
mad['prestaciones_pct'] = mad[PENSIONES] + mad[PARO]
mad['indice_vulnerabilidad'] = (
    (1 - norm(mad[RENTA_NETA_HOGAR])) + norm(mad[GINI]) + norm(mad[MAYORES65]) + norm(mad['prestaciones_pct'])
) / 4 * 100
mad['poblacion_extranjera_pct'] = 100 - mad[ESPANOLA]

# geometría unida (para los mapas) — incluye las derivadas
gdf = maps.unir_distritos_madrid(ZIP, mad)
print(len(mad), 'distritos ·', mad.shape[1], 'columnas (incluidas las derivadas)')


## 1. Renta por distrito
Renta neta media por hogar. Mapa + ranking.


In [ ]:
maps.mapa_folium(gdf, RENTA_NETA_HOGAR,
                 guardar_html=str(config.OUTPUTS / 'mapa_renta.html'))


In [ ]:
d = mad.sort_values(RENTA_NETA_HOGAR)
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(d['nombre_distrito'], d[RENTA_NETA_HOGAR], color='#4361ee')
ax.set_xlabel(RENTA_NETA_HOGAR + ' (€/año)'); ax.set_title('Renta por distrito')
plt.tight_layout(); plt.show()
print('Ratio rico/pobre:', round(d[RENTA_NETA_HOGAR].max()/d[RENTA_NETA_HOGAR].min(), 2), 'veces')


**Conclusión:** existe una clara brecha territorial — los distritos del noroeste (Chamartín, Salamanca, Chamberí) casi duplican la renta de los del sur (Puente de Vallecas, Usera, Villaverde).


## 2. Renta vs desigualdad (Gini)
¿Los distritos más ricos son más desiguales por dentro?


In [ ]:
fig, ax = plt.subplots(figsize=(8.5,6.5))
sc = ax.scatter(mad[RENTA_NETA_HOGAR], mad[GINI], s=mad[POB]/2500, c=mad[SALARIO], cmap='RdYlGn')
for _, r in mad.iterrows(): ax.annotate(r['nombre_distrito'], (r[RENTA_NETA_HOGAR], r[GINI]), fontsize=6, alpha=.7)
ax.set_xlabel(RENTA_NETA_HOGAR); ax.set_ylabel(GINI)
plt.colorbar(sc, label='% salario'); plt.title('Renta vs desigualdad'); plt.tight_layout(); plt.show()
print('Correlación renta↔Gini:', round(mad[RENTA_NETA_HOGAR].corr(mad[GINI]), 2))


**Conclusión:** correlación positiva renta↔Gini: los distritos ricos concentran más desigualdad interna (conviven rentas muy altas y medias).


## 3. ¿De qué se vive? Fuente de los ingresos
% de la renta que viene de salario, pensiones, paro y otras.


In [ ]:
fuentes = [(SALARIO,'Salario'), (PENSIONES,'Pensiones'), (PARO,'Paro'),
           (OTRAS_PREST,'Otras prest.'), (OTROS_ING,'Otros')]
d = mad.sort_values(SALARIO)
bottom = np.zeros(len(d)); fig, ax = plt.subplots(figsize=(10,8))
for col, lab in fuentes:
    ax.barh(d['nombre_distrito'], d[col], left=bottom, label=lab)
    bottom = bottom + d[col].values
ax.set_xlabel('% sobre renta bruta'); ax.legend(loc='lower right', fontsize=8)
ax.set_title('Fuente de los ingresos por distrito'); plt.tight_layout(); plt.show()


**Conclusión:** en los distritos de menor renta pesa más la **pensión** y las **prestaciones**; en los de mayor renta, una parte mayor viene de **otros ingresos** (rentas del capital), no solo del salario.


## 4. Peso de impuestos *(variable derivada)*
`(renta bruta − renta neta) / renta bruta`: cuánto se descuenta de la renta bruta.


In [ ]:
maps.mapa_folium(gdf, 'peso_impuestos_pct',
                 guardar_html=str(config.OUTPUTS / 'mapa_impuestos.html'))


In [ ]:
d = mad.sort_values('peso_impuestos_pct')
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(d['nombre_distrito'], d['peso_impuestos_pct'], color='#b5179e')
ax.set_xlabel('% de la renta bruta en impuestos/cotizaciones'); plt.tight_layout(); plt.show()


**Conclusión:** el descuento efectivo (IRPF + cotizaciones) es mayor en los distritos de renta alta — coherente con la progresividad del sistema fiscal.


## 5. Envejecimiento
% de población de 65 y más años y edad media.


In [ ]:
maps.mapa_folium(gdf, MAYORES65,
                 guardar_html=str(config.OUTPUTS / 'mapa_mayores65.html'))


**Conclusión:** los distritos consolidados (centro y norte) están más envejecidos; la periferia (Villa de Vallecas, Vicálvaro) es más joven.


## 6. Cómo se habita
Hogares unipersonales y tamaño medio del hogar.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13,7))
for ax, col, t in [(axs[0], UNIPERSONAL, '% hogares unipersonales'),
                   (axs[1], 'demo_Tamaño medio del hogar', 'Tamaño medio del hogar')]:
    d = mad.sort_values(col); ax.barh(d['nombre_distrito'], d[col]); ax.set_title(t)
plt.tight_layout(); plt.show()


**Conclusión:** en el centro dominan los **hogares de una persona**; en la periferia, hogares más grandes (familias).


## 7. Población extranjera *(derivada: 100 − % española)*


In [ ]:
maps.mapa_folium(gdf, 'poblacion_extranjera_pct',
                 guardar_html=str(config.OUTPUTS / 'mapa_extranjera.html'))


**Conclusión:** la población migrante no se reparte de forma homogénea: se concentra más en unos distritos que en otros.


## 8. Brecha rico/pobre *(derivada)*
Ratio de renta entre el distrito más rico y el más pobre, y P80/P20 del INE.


In [ ]:
ratio = mad[RENTA_NETA_HOGAR].max() / mad[RENTA_NETA_HOGAR].min()
print(f'El distrito más rico tiene {ratio:.2f}x la renta del más pobre.')
d = mad.sort_values(P80P20)
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(d['nombre_distrito'], d[P80P20], color='#7209b7')
ax.set_xlabel('Distribución de la renta P80/P20 (INE)'); plt.tight_layout(); plt.show()


**Conclusión:** la desigualdad territorial es notable; el ratio P80/P20 muestra además la desigualdad interna de cada distrito.


## 9. Índice de vulnerabilidad *(derivada compuesta)*
Combina (igual peso): renta baja + desigualdad (Gini) + envejecimiento + dependencia de prestaciones. 0 = menos vulnerable, 100 = más. Ver método en `04_procesos_data/04_analisis/indices_compuestos.md`.


In [ ]:
maps.mapa_folium(gdf, 'indice_vulnerabilidad',
                 guardar_html=str(config.OUTPUTS / 'mapa_vulnerabilidad.html'))


In [ ]:
d = mad.sort_values('indice_vulnerabilidad', ascending=False)
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(d['nombre_distrito'], d['indice_vulnerabilidad'], color='#cf1d35')
ax.set_xlabel('Índice de vulnerabilidad (0–100)'); plt.tight_layout(); plt.show()
print('Top 3 más vulnerables:', list(d['nombre_distrito'].head(3)))


**Conclusión:** el índice resume en una cifra qué distritos acumulan más factores de vulnerabilidad (renta baja + desigualdad + envejecimiento + dependencia de prestaciones). *Limitación:* pesos iguales por simplicidad; conviene un análisis de sensibilidad.


## 10. Mapa de correlaciones
Qué variables van juntas (rojo = relación positiva, azul = negativa).


In [ ]:
num = mad.select_dtypes('number')
corr = num.corr()
fig, ax = plt.subplots(figsize=(11,9))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=6)
plt.colorbar(im, fraction=0.046); plt.title('Correlaciones'); plt.tight_layout(); plt.show()


**Conclusión:** se ven bloques de variables relacionadas (p. ej. renta alta ↔ menor dependencia de pensiones; envejecimiento ↔ hogares unipersonales).


## 11. Perfiles de distrito *(opcional · clustering K-Means)*
Agrupa los distritos en perfiles socioeconómicos similares.


In [ ]:
try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    feats = [RENTA_NETA_HOGAR, GINI, MAYORES65, SALARIO, 'indice_vulnerabilidad']
    X = StandardScaler().fit_transform(mad[feats])
    mad['cluster'] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X).astype(str)
    gdfc = maps.unir_distritos_madrid(ZIP, mad)
    maps.mapa_choropleth(gdfc, 'cluster', 'Perfiles de distrito (clusters)', cmap='Set2')
    plt.show()
    print(mad.groupby('cluster')['nombre_distrito'].apply(list))
except Exception as e:
    print('Clustering omitido (¿falta scikit-learn?):', e)


**Conclusión:** los distritos se agrupan en perfiles reconocibles (p. ej. *centro rico*, *periferia joven*, *sur vulnerable*), útiles para segmentar el análisis.


## Ficha de conclusiones

- **Brecha territorial:** el distrito más rico casi duplica (o más) la renta del más pobre.
- **Desigualdad:** los distritos ricos son los más desiguales por dentro (renta↔Gini positiva).
- **Ingresos:** sur más dependiente de pensiones/prestaciones; norte con más rentas del capital.
- **Fiscalidad:** mayor descuento efectivo en los distritos de renta alta (progresividad).
- **Demografía:** centro envejecido y unipersonal; periferia más joven y familiar.
- **Vulnerabilidad:** el índice compuesto identifica los distritos que acumulan desventajas.

**Limitaciones:** dato por distrito (no por persona, ojo a la falacia ecológica); año 2023; correlación ≠ causalidad; el índice de vulnerabilidad usa pesos iguales (revisable).

**Siguiente:** añadir **vivienda** (alquiler/compra) para el *esfuerzo real* — ver `docs/PLAN_VIVIENDA.md`.
